# 00 · PyTorch 环境准备与快速上手

> 本套教程目标：从零掌握 PyTorch 与深度学习，全部代码可直接运行。
> 学习顺序：按编号 00 → 12 依次进行，**不要跳**，后面的实战依赖前面的基础。

## 0.1 你将学到什么

| 模块 | 内容 | 对应文件 |
|---|---|---|
| 基础 | 张量、自动求导、nn.Module | 01–03 |
| 数据 | Dataset / DataLoader / transforms | 04 |
| 训练 | 损失函数、优化器、训练循环 | 05 |
| 实战 | 全连接、CNN、RNN、Transformer、迁移学习 | 06–10 |
| 进阶 | 正则化、初始化、AMP、部署与性能 | 11–12 |

## 0.2 环境要求

- Python ≥ 3.9（推荐 3.10/3.11）
- PyTorch ≥ 2.0
- 可选：NVIDIA GPU + CUDA（CPU 也能学完本套教程，只是慢）

## 0.3 安装命令

根据你的显卡选择。在终端（不是本 notebook）执行：

```bash
# 只有 CPU
pip install torch torchvision torchaudio

# CUDA 12.1（NVIDIA 显卡，推荐先装好驱动）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 国内加速（清华源）
pip install torch torchvision torchaudio -i https://pypi.tuna.tsinghua.edu.cn/simple
```

其它依赖（本教程会用到）：

```bash
pip install numpy matplotlib pandas scikit-learn tqdm jupyterlab
```

## 0.4 验证安装

运行下面两个代码块，确认版本与设备。

In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import matplotlib

print("PyTorch       :", torch.__version__)
print("torchvision   :", torchvision.__version__)
print("NumPy         :", np.__version__)
print("Matplotlib    :", matplotlib.__version__)
print("CUDA 可用     :", torch.cuda.is_available())
print("CUDA 版本     :", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU 名称      :", torch.cuda.get_device_name(0))
    print("GPU 数量      :", torch.cuda.device_count())

### 关于 device（设备）

PyTorch 中张量和模型都需要放在某个设备上：`cpu` 或 `cuda`（GPU）。
下面定义一个 `device` 变量，后面所有教程统一使用：

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("本次运行使用设备：", device)

# 在 GPU 上创建张量
x = torch.tensor([1.0, 2.0, 3.0], device=device)
print("张量所在设备：", x.device)
print(x)

## 0.5 第一个完整例子：线性回归

不用任何封装，用最原始的方式走一遍**深度学习的标准五步**：

1. **准备数据**
2. **定义模型**
3. **定义损失函数**
4. **定义优化器**
5. **训练循环**（前向 → 计算损失 → 反向传播 → 更新参数）

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ---------- 1. 准备数据 ----------
# 真实关系: y = 3x + 2，加一点噪声
X = torch.rand(100, 1) * 10 - 5          # 100 个样本，每个 1 个特征
noise = torch.randn(100, 1) * 0.3
y = 3.0 * X + 2.0 + noise

print("X 形状:", X.shape, " y 形状:", y.shape)

# ---------- 2. 定义模型 ----------
model = nn.Linear(1, 1)                   # 一个线性层: y = wx + b
print("初始参数 w, b =", model.weight.item(), model.bias.item())

# ---------- 3. 损失函数 ----------
criterion = nn.MSELoss()                  # 均方误差

# ---------- 4. 优化器 ----------
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# ---------- 5. 训练循环 ----------
for epoch in range(200):
    # 前向传播
    y_pred = model(X)
    # 计算损失
    loss = criterion(y_pred, y)
    # 清空上一步的梯度（必须！否则会累加）
    optimizer.zero_grad()
    # 反向传播，计算梯度
    loss.backward()
    # 更新参数
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | loss = {loss.item():.4f}")

print("-" * 40)
print(f"学到的 w = {model.weight.item():.3f}  (真实值 3.0)")
print(f"学到的 b = {model.bias.item():.3f}  (真实值 2.0)")

看到了吗？200 次迭代后，模型把 `w≈3`、`b≈2` 学出来了。

**记住这个五步循环，它贯穿整个深度学习**：

```
optimizer.zero_grad()   # 1) 清空旧梯度
y_pred = model(x)       # 2) 前向传播
loss = criterion(y_pred, y_true)  # 3) 计算损失
loss.backward()         # 4) 反向传播
optimizer.step()        # 5) 更新参数
```

## 0.6 常见环境问题排查

| 问题 | 解决办法 |
|---|---|
| `torch.cuda.is_available()` 为 False | 装错版本（装了 CPU 版）；或驱动过旧，更新显卡驱动 |
| 显存不足 OOM | 减小 `batch_size`；用 `torch.cuda.empty_cache()` |
| Windows 上报 `num_workers` 错误 | DataLoader 里把 `num_workers` 设为 0 |
| Jupyter 找不到 torch | kernel 选错，或没在同一个 Python 环境里安装 |

## 0.7 下一步

进入 `01_张量Tensor基础.ipynb`。